In [7]:
%pip install qiskit qiskit-aer qiskit-algorithms qiskit-nature numpy scipy matplotlib networkx pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
"""
================================================================================
PROYECTO: IA-Energy-Alternatives
MÓDULO 01: Diseño Cuántico de Fluido Dieléctrico Libre de PFAS (Vector A)
GRUPO: Dark Shadows Group
================================================================================

JUSTIFICACIÓN TÉCNICA Y FÍSICA (¿POR QUÉ HACEMOS ESTO?):

1. EL PROBLEMA DE INFRAESTRUCTURA:
   Estamos ejecutando Python 3.14 sobre Windows nativo. El motor ab-initio estándar 
   (PySCFDriver) requiere compilar librerías de C/Fortran vinculadas a binarios 
   POSIX (BLAS/LAPACK). Windows no resuelve estas dependencias nativas en el Path,
   lo que genera un colapso de CMake (MissingOptionalLibraryError).

2. LA SOLUCIÓN (BYPASS ANALÍTICO):
   En lugar de frenar el proyecto para reconfigurar compiladores de C++, separamos 
   la química clásica de la computación cuántica. PySCF únicamente sirve para 
   calcular integrales de superposición orbital. Nosotros salteamos ese paso 
   inyectando directamente los tensores de integrales de un cuerpo (h1) y de dos 
   cuerpos (h2) que modelan un dipolo electrónico polarizado (enlace C-F).

3. OBJETIVO DEL BLOQUE:
   Instanciar el Hamiltoniano electrónico molecular desde integrales crudas, 
   aplicar la transformación isomórfica de Jordan-Wigner para mapear los 
   fermiones a operadores de espín (Pauli), y dejar el problema listo para 
   ser minimizado por el Variational Quantum Eigensolver (VQE).
================================================================================
"""

import sys
import numpy as np

# Inyección del path de configuración transversal del grupo
sys.path.append('..')
from config.quantum_settings import get_local_quantum_backend, get_optimizer

from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.hamiltonians import ElectronicEnergy
from qiskit_algorithms import VQE

# 1. PARAMETRIZACIÓN DEL ESPACIO ACTIVO (Enlace Carbono-Flúor simulado)
num_spatial_orbitals = 2
num_particles = (1, 1)  # 1 electrón con espín Alfa (+1/2), 1 con espín Beta (-1/2)

# 2. CONSTRUCCIÓN DEL TENSOR DE ENERGÍA ELECTRÓNICA
# h1: Energía cinética + atracción nuclear de los electrones individuales
h1_matrix = np.array([
    [-1.25,  0.20], 
    [ 0.20, -0.85]
])

# h2: Repulsión culombiana electrón-electrón (tensor 4D)
h2_matrix = np.zeros((2, 2, 2, 2))
h2_matrix[0, 0, 1, 1] = 0.18
h2_matrix[1, 1, 0, 0] = 0.18

# 3. ENSAMBLADO DEL HAMILTONIANO FERMIÓNICO
electronic_energy = ElectronicEnergy.from_raw_integrals(h1_matrix, h2_matrix)
hamiltonian = electronic_energy.second_q_op()

# 4. MAPEO A QUBITS (Jordan-Wigner)
mapper = JordanWignerMapper()
qubit_op = mapper.map(hamiltonian)

print(f"[OK] Bypass analítico ejecutado con éxito.")
print(f"[OK] Operador de Pauli generado: {qubit_op.num_qubits} qubits listos para QPU/Simulador.")

[OK] Bypass analítico ejecutado con éxito.
[OK] Operador de Pauli generado: 4 qubits listos para QPU/Simulador.
